# VietMedBridge — 02: Trích văn bản và chia đoạn có provenance

Chạy sau notebook 01. Trích HTML/XML/PDF thành source_text, rồi tạo child
và parent bằng tokenizer BGE-M3 fast. Chỉ tải tokenizer; runtime CPU đủ.
Mỗi span được kiểm tra bằng lát cắt source_text[start_char:end_char].

source_text là văn bản nguồn đã trích từ snapshot; offset tính bằng ký tự
Unicode của chuỗi Python, không phải vị trí byte trong HTML/PDF.
retrieval_text được chuẩn hóa riêng. Tham số 180/40/512 là cấu hình khởi đầu
để thử nghiệm, chưa phải cấu hình đã tối ưu Chunk F2.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "có thể chọn Runtime Version 2026.07 (Python 3.12)."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả ba notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True
).strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(CHECKOUT) + "[notebook]"],
    check=True,
)
from vietmedbridge.artifacts import atomic_json, runtime_versions
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT})
atomic_json(DATA_ROOT / "runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Chọn crawl run và ghim revision tokenizer


In [ ]:
from vietmedbridge.artifacts import read_json, atomic_json
from vietmedbridge.chunks import ChunkConfig, load_bge_tokenizer
from vietmedbridge.build import build_corpus
from vietmedbridge.crawl import completed_parts

CRAWL_RUN = "smoke-v1"  # đổi thành corpus-v1 khi xử lý run lớn
BUILD_RUN = "canonical-v1"
MAX_NEW_SHARDS = None
TOKENIZER_REVISION = None
tokenizer_lock_path = DATA_ROOT / "tokenizer_lock.json"
tokenizer_lock = read_json(tokenizer_lock_path) if tokenizer_lock_path.exists() else {}
requested_revision = TOKENIZER_REVISION or tokenizer_lock.get("revision")
TOKENIZER, TOKENIZER_SPEC = load_bge_tokenizer(requested_revision)
if not tokenizer_lock or TOKENIZER_REVISION:
    atomic_json(tokenizer_lock_path, TOKENIZER_SPEC)
print("Tokenizer:", TOKENIZER_SPEC)
print("Complete raw shards:", len(completed_parts(DATA_ROOT / "crawl" / CRAWL_RUN)))


## 2. Tạo Parquet cho documents, children, parents và failures

Parser HTML dùng trafilatura; XML xử lý abstract/body; PDF lấy text theo thứ
tự trang. PDF scan không có text và trang challenge bị ghi lỗi để xử lý tiếp.
Các ID chính thức có cùng nội dung vẫn được giữ riêng.
Parent là cửa sổ theo token trong cùng tài liệu; phân tách section chuyên biệt
và OCR chưa được triển khai ở giai đoạn này.


In [ ]:
CHUNK_CONFIG = ChunkConfig(
    child_tokens=180, overlap_tokens=40, parent_tokens=512, snap_sentences=True,
)
BUILD = build_corpus(
    DATA_ROOT / "crawl" / CRAWL_RUN, DATA_ROOT / "processed",
    TOKENIZER, TOKENIZER_SPEC, run_name=BUILD_RUN, config=CHUNK_CONFIG,
    work_dir=WORK_DIR, max_shards=MAX_NEW_SHARDS,
)
print(json.dumps({
    key: BUILD[key] for key in (
        "run_name", "snapshot_sha256", "counts", "selected_shards", "available_crawl_shards"
    )
}, ensure_ascii=False, indent=2))


## 3. Xem một tài liệu và kiểm tra span đã lưu


In [ ]:
import pyarrow.parquet as pq
from vietmedbridge.build import verify_spans

BUILD_DIR = DATA_ROOT / "processed" / BUILD_RUN
document_preview = None
for part in BUILD["parts"]:
    documents = pq.ParquetFile(BUILD_DIR / part["files"]["documents"]["path"])
    if documents.metadata.num_rows:
        document_preview = next(documents.iter_batches(batch_size=1)).to_pylist()[0]
        break
if document_preview is not None:
    doc_id = document_preview["doc_id"]
    children = pq.read_table(
        BUILD_DIR / part["files"]["children"]["path"],
        filters=[("doc_id", "=", doc_id)],
    ).to_pylist()
    parents = pq.read_table(
        BUILD_DIR / part["files"]["parents"]["path"],
        filters=[("doc_id", "=", doc_id)],
    ).to_pylist()
    verify_spans(document_preview, children, parents)
    print("Verified doc_id:", doc_id, "| title:", document_preview["title"])
    print("Language hint:", document_preview["language"], document_preview["language_method"])
    if children:
        print("Child source text:", children[0]["text"][:1500])
    if parents:
        print("Parent source text:", parents[0]["text"][:2500])
else:
    print("Chưa có tài liệu trích được. Xem failures và crawl summary.")


## 4. Các artifact dùng cho giai đoạn retrieval sau này

Chỉ đọc danh sách file trong build.json/snapshot manifest. Không glob toàn bộ
processed folder, vì nơi này giữ cả các attempt cũ để rollback.
documents giữ source_text và hash; children/parents giữ offset cùng raw source ID.
failures giữ cả lỗi crawl và lỗi parser. Language chỉ là hint phục vụ audit,
chưa được dùng làm hard gate.

Notebook chưa tạo embedding/index và chưa tính F2. Query công khai hiện không
có reference labels; BGE-token windows cũng chưa thay thế scorer chính thức.
Raw corpus và artifact được lưu trên Drive, không đưa vào GitHub.
